<a href="https://colab.research.google.com/github/RamaraoD423/GenAI_L3---Bronze/blob/main/cudaNVIDIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Configuration
DATA_DIR = "./data"
DB_DIR = "./vector_store"
MODEL_NAME = "llama3.2"
EMBEDDING_MODEL = "nomic-embed-text"

def initialize_vector_store():
    """Loads all PDFs from directory, chunks them, and saves to Chroma DB."""
    print("Loading PDF documents from directory...")
    loader = PyPDFDirectoryLoader(DATA_DIR)
    docs = loader.load()

    if not docs:
        raise ValueError(f"No PDF files found in {DATA_DIR}. Please add at least 3 PDFs.")

    print(f"Loaded {len(docs)} total pages. Splitting into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks = text_splitter.split_documents(docs)

    print("Generating local embeddings and creating vector store...")
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_DIR
    )
    print("Vector database built and saved locally!")
    return vector_store

def get_rag_chain(vector_store):
    """Builds the LCEL RAG chain using Llama 3.2."""
    retriever = vector_store.as_retriever(search_kwargs={"k": 4})
    llm = ChatOllama(model=MODEL_NAME)

    # Prompt template enforcing strict grounding to context
    template = """Answer the question strictly using only the provided context below.
If the answer cannot be found in the context, say "I cannot find the answer in the provided documents."

Context:
{context}

Question: {question}
"""
    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain

def main():
    # Ensure data folder exists
    if not os.path.exists(DATA_DIR):
        os.makedirs(DATA_DIR)
        print(f"Created '{DATA_DIR}' folder. Please place your 3+ PDF documents inside it and rerun.")
        return

    # Load or create vector database
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
    if os.path.exists(DB_DIR) and os.listdir(DB_DIR):
        print("Loading existing vector store from disk...")
        vector_store = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
    else:
        vector_store = initialize_vector_store()

    rag_chain = get_rag_chain(vector_store)

    print("\n--- RAG Application Running (Llama 3.2 via Ollama) ---")
    print("Ask questions about your PDFs. Type 'exit' to quit.\n")

    while True:
        query = input("Query: ")
        if query.lower() == 'exit':
            break
        if not query.strip():
            continue

        print("\nThinking...")
        response = rag_chain.invoke(query)
        print(f"\nAnswer:\n{response}\n" + "-"*50)

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Configuration
DATA_DIR = "./data"
DB_DIR = "./vector_store"
MODEL_NAME = "llama3.2"
EMBEDDING_MODEL = "nomic-embed-text"

def initialize_vector_store():
    """Loads all PDFs from directory, chunks them, and saves to Chroma DB."""
    print("Loading PDF documents from directory...")
    loader = PyPDFDirectoryLoader(DATA_DIR)
    docs = loader.load()

    if not docs:
        raise ValueError(f"No PDF files found in {DATA_DIR}. Please add at least 3 PDFs.")

    print(f"Loaded {len(docs)} total pages. Splitting into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks = text_splitter.split_documents(docs)

    print("Generating local embeddings and creating vector store...")
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_DIR
    )
    print("Vector database built and saved locally!")
    return vector_store

def get_rag_chain(vector_store):
    """Builds the LCEL RAG chain using Llama 3.2."""
    retriever = vector_store.as_retriever(search_kwargs={"k": 4})
    llm = ChatOllama(model=MODEL_NAME)

    # Prompt template enforcing strict grounding to context
    template = """Answer the question strictly using only the provided context below.
If the answer cannot be found in the context, say "I cannot find the answer in the provided documents."

Context:
{context}

Question: {question}
"""
    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain

def main():
    # Ensure data folder exists
    if not os.path.exists(DATA_DIR):
        os.makedirs(DATA_DIR)
        print(f"Created '{DATA_DIR}' folder. Please place your 3+ PDF documents inside it and rerun.")
        return

    # Load or create vector database
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
    if os.path.exists(DB_DIR) and os.listdir(DB_DIR):
        print("Loading existing vector store from disk...")
        vector_store = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
    else:
        vector_store = initialize_vector_store()

    rag_chain = get_rag_chain(vector_store)

    print("\n--- RAG Application Running (Llama 3.2 via Ollama) ---")
    print("Ask questions about your PDFs. Type 'exit' to quit.\n")

    while True:
        query = input("Query: ")
        if query.lower() == 'exit':
            break
        if not query.strip():
            continue

        print("\nThinking...")
        response = rag_chain.invoke(query)
        print(f"\nAnswer:\n{response}\n" + "-"*50)

if __name__ == "__main__":
    main()

In [ ]:
#include "llama.h"
#include <iostream>
#include <vector>
#include <string>

int main(int argc, char **argv) {
    if (argc < 3) {
        std::cerr << "Usage: " << argv[0] << " <path_to_model.gguf> <prompt>\n";
        return 1;
    }

    std::string model_path = argv[1];
    std::string prompt = argv[2];

    // 1. Initialize the llama backend
    llama_backend_init();

    // 2. Load model parameters
    llama_model_params model_params = llama_model_default_params();
    // To offload all layers to the NVIDIA GPU via CUDA, set n_gpu_layers high (e.g., 99)
    model_params.n_gpu_layers = 99;

    std::cout << "Loading model from " << model_path << " (with CUDA acceleration)..." << std::endl;
    llama_model *model = llama_model_load_from_file(model_path.c_str(), model_params);
    if (!model) {
        std::cerr << "Error: Unable to load model." << std::endl;
        return 1;
    }

    // 3. Initialize context parameters
    llama_context_params ctx_params = llama_context_default_params();
    ctx_params.n_ctx = 2048; // Context window size

    llama_context *ctx = llama_init_from_model(model, ctx_params);
    if (!ctx) {
        std::cerr << "Error: Unable to create context." << std::endl;
        llama_model_free(model);
        return 1;
    }

    // 4. Tokenize the prompt
    const llama_vocab *vocab = llama_model_get_vocab(model);

    // Determine token count estimation
    std::vector<llama_token> tokens(prompt.size() + 3);
    int n_tokens = llama_vocab_tokenize(vocab, prompt.c_str(), prompt.size(), tokens.data(), tokens.size(), true, true);
    if (n_tokens < 0) {
        tokens.resize(-n_tokens);
        n_tokens = llama_vocab_tokenize(vocab, prompt.c_str(), prompt.size(), tokens.data(), tokens.size(), true, true);
    }
    tokens.resize(n_tokens);

    // 5. Prepare batch for evaluation
    llama_batch batch = llama_batch_init(512, 0, 1);
    for (int i = 0; i < n_tokens; i++) {
        llama_batch_add(batch, tokens[i], i, { 0 }, false);
    }
    batch.logits[batch.n_tokens - 1] = true; // Enable logits for the last token

    // Run initial prompt evaluation on GPU
    if (llama_decode(ctx, batch) != 0) {
        std::cerr << "llama_decode failed during prompt processing." << std::endl;
        return 1;
    }

    // 6. Generate tokens loop
    std::cout << "\n--- Response ---\n" << prompt;

    // Simple greedy sampler setup (or use llama_sampler API for advanced sampling)
    // For brevity, we grab the logits of the last token
    int n_cur = batch.n_tokens;
    int n_predict = 128; // Max tokens to generate

    for (int i = 0; i < n_predict; i++) {
        auto *logits = llama_get_logits_ith(ctx, batch.n_tokens - 1);
        auto n_vocab = llama_vocab_n_tok(vocab);

        // Find the most likely token (greedy)
        llama_token new_token_id = 0;
        float max_logit = logits[0];
        for (int v = 1; v < n_vocab; v++) {
            if (logits[v] > max_logit) {
                max_logit = logits[v];
                new_token_id = v;
            }
        }

        // Check for end-of-stream token
        if (llama_vocab_is_eog(vocab, new_token_id)) {
            break;
        }

        // Convert token to text piece and print
        char buf[256];
        int n = llama_vocab_token_to_text(vocab, new_token_id, buf, sizeof(buf), 0);
        if (n > 0) {
            std::cout << std::string(buf, n) << std::flush;
        }

        // Prepare next batch with the single generated token
        llama_batch_clear(batch);
        llama_batch_add(batch, new_token_id, n_cur, { 0 }, true);

        if (llama_decode(ctx, batch) != 0) {
            std::cerr << "llama_decode failed during generation." << std::endl;
            break;
        }
        n_cur++;
    }
    std::cout << "\n----------------\n";

    // 7. Cleanup
    llama_batch_free(batch);
    llama_free(ctx);
    llama_model_free(model);
    llama_backend_free();

    return 0;
}

In [ ]:
#include "llama.h"
#include <iostream>
#include <vector>
#include <string>

int main(int argc, char **argv) {
    if (argc < 3) {
        std::cerr << "Usage: " << argv[0] << " <path_to_model.gguf> <prompt>\n";
        return 1;
    }

    std::string model_path = argv[1];
    std::string prompt = argv[2];

    // 1. Initialize the llama backend
    llama_backend_init();

    // 2. Load model parameters
    llama_model_params model_params = llama_model_default_params();
    // To offload all layers to the NVIDIA GPU via CUDA, set n_gpu_layers high (e.g., 99)
    model_params.n_gpu_layers = 99;

    std::cout << "Loading model from " << model_path << " (with CUDA acceleration)..." << std::endl;
    llama_model *model = llama_model_load_from_file(model_path.c_str(), model_params);
    if (!model) {
        std::cerr << "Error: Unable to load model." << std::endl;
        return 1;
    }

    // 3. Initialize context parameters
    llama_context_params ctx_params = llama_context_default_params();
    ctx_params.n_ctx = 2048; // Context window size

    llama_context *ctx = llama_init_from_model(model, ctx_params);
    if (!ctx) {
        std::cerr << "Error: Unable to create context." << std::endl;
        llama_model_free(model);
        return 1;
    }

    // 4. Tokenize the prompt
    const llama_vocab *vocab = llama_model_get_vocab(model);

    // Determine token count estimation
    std::vector<llama_token> tokens(prompt.size() + 3);
    int n_tokens = llama_vocab_tokenize(vocab, prompt.c_str(), prompt.size(), tokens.data(), tokens.size(), true, true);
    if (n_tokens < 0) {
        tokens.resize(-n_tokens);
        n_tokens = llama_vocab_tokenize(vocab, prompt.c_str(), prompt.size(), tokens.data(), tokens.size(), true, true);
    }
    tokens.resize(n_tokens);

    // 5. Prepare batch for evaluation
    llama_batch batch = llama_batch_init(512, 0, 1);
    for (int i = 0; i < n_tokens; i++) {
        llama_batch_add(batch, tokens[i], i, { 0 }, false);
    }
    batch.logits[batch.n_tokens - 1] = true; // Enable logits for the last token

    // Run initial prompt evaluation on GPU
    if (llama_decode(ctx, batch) != 0) {
        std::cerr << "llama_decode failed during prompt processing." << std::endl;
        return 1;
    }

    // 6. Generate tokens loop
    std::cout << "\n--- Response ---\n" << prompt;

    // Simple greedy sampler setup (or use llama_sampler API for advanced sampling)
    // For brevity, we grab the logits of the last token
    int n_cur = batch.n_tokens;
    int n_predict = 128; // Max tokens to generate

    for (int i = 0; i < n_predict; i++) {
        auto *logits = llama_get_logits_ith(ctx, batch.n_tokens - 1);
        auto n_vocab = llama_vocab_n_tok(vocab);

        // Find the most likely token (greedy)
        llama_token new_token_id = 0;
        float max_logit = logits[0];
        for (int v = 1; v < n_vocab; v++) {
            if (logits[v] > max_logit) {
                max_logit = logits[v];
                new_token_id = v;
            }
        }

        // Check for end-of-stream token
        if (llama_vocab_is_eog(vocab, new_token_id)) {
            break;
        }

        // Convert token to text piece and print
        char buf[256];
        int n = llama_vocab_token_to_text(vocab, new_token_id, buf, sizeof(buf), 0);
        if (n > 0) {
            std::cout << std::string(buf, n) << std::flush;
        }

        // Prepare next batch with the single generated token
        llama_batch_clear(batch);
        llama_batch_add(batch, new_token_id, n_cur, { 0 }, true);

        if (llama_decode(ctx, batch) != 0) {
            std::cerr << "llama_decode failed during generation." << std::endl;
            break;
        }
        n_cur++;
    }
    std::cout << "\n----------------\n";

    // 7. Cleanup
    llama_batch_free(batch);
    llama_free(ctx);
    llama_model_free(model);
    llama_backend_free();

    return 0;
}